# Stanford AI Vibrancy Tool — Visualizations

Visualizations of `ai_country_scores.csv` exported from the **Stanford AI Vibrancy Tool**.

The dataset contains a **Total Score** and 7 sub-dimensions for 8 countries (US, CN, UK, CA, DE, FR, JP, IT):

- **R&D** — research and development output
- **Responsible AI** — ethics, governance frameworks
- **Economy** — investment and adoption
- **Talent** — AI workforce
- **Policy and Governance** — legislation and government action
- **Public Opinion** — societal attitudes
- **Infrastructure** — compute and data centers


## 1. Setup & Load

In [ ]:
from pathlib import Path
import sys

repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / 'data').is_dir() and (p / 'src').is_dir()
)
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from ai_policy.common import find_repo_root

plt.rcParams['axes.unicode_minus'] = False

repo_root = find_repo_root(Path.cwd())
csv_path = repo_root / 'data' / 'AI vibrancy tool screen shot' / 'ai_country_scores.csv'

assert csv_path.exists(), f'{csv_path} not found'

df = pd.read_csv(csv_path)
df = df.set_index('Country')
print(f'Loaded: {df.shape[0]} countries × {df.shape[1]} columns')
df



## 2. Country Colors

In [ ]:
from ai_policy.common import COUNTRY_COLORS as COLORS

DIMENSIONS = [c for c in df.columns if c != 'Total Score']
print('Dimensions:', DIMENSIONS)



## 3. Descriptive Statistics for AI Vibrancy Metrics

This table reports count, mean, median, standard deviation, minimum, and maximum for the Stanford AI Vibrancy total score and component metrics across the eight selected countries.


In [ ]:
from ai_policy.stats import descriptive_statistics

vibrancy_numeric_cols = ['Total Score'] + DIMENSIONS
vibrancy_descriptive_stats = descriptive_statistics(df, vibrancy_numeric_cols)

vibrancy_stats_path = repo_root / 'outputs' / 'descriptive_statistics_ai_vibrancy.csv'
vibrancy_stats_path.parent.mkdir(parents=True, exist_ok=True)
vibrancy_descriptive_stats.to_csv(vibrancy_stats_path)
print(f'Saved: {vibrancy_stats_path}')
vibrancy_descriptive_stats


## 3. Total Score Ranking

In [ ]:
ranked = df.sort_values('Total Score', ascending=True)

fig, ax = plt.subplots(figsize=(11, 5))
colors = [COLORS[c] for c in ranked.index]
ax.barh(ranked.index, ranked['Total Score'], color=colors)
ax.set_xlabel('Total Score')
ax.set_title('Stanford AI Vibrancy Tool — Total Score', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

for i, v in enumerate(ranked['Total Score']):
    ax.text(v + 0.6, i, f'{v:.2f}', va='center', fontsize=10)

ax.set_xlim(0, ranked['Total Score'].max() * 1.1)
plt.tight_layout()
plt.show()

print('\nRanking:')
df.sort_values('Total Score', ascending=False)[['Total Score']]


## 4. Stacked Bar — Contribution of Each Dimension to Total Score

This shows what's driving each country's overall vibrancy.


In [ ]:
ranked_by_total = df.sort_values('Total Score', ascending=False)
comp = ranked_by_total[DIMENSIONS]

# Distinct palette for the 7 dimensions
dim_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728',
              '#9467bd', '#8c564b', '#e377c2']

fig, ax = plt.subplots(figsize=(13, 6))
bottom = np.zeros(len(comp))
for dim, color in zip(DIMENSIONS, dim_colors):
    ax.bar(comp.index, comp[dim], bottom=bottom, label=dim, color=color, edgecolor='white', linewidth=0.5)
    bottom += comp[dim].values

ax.set_ylabel('Score Contribution')
ax.set_title('Stanford AI Vibrancy — Dimension Contributions to Total Score',
             fontsize=13, fontweight='bold')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=20, ha='right')

# Total label on top of each stack
for i, (country, total) in enumerate(zip(comp.index, ranked_by_total['Total Score'])):
    ax.text(i, total + 1, f'{total:.1f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


## 5. Heatmap — Country × Dimension

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(comp.values, aspect='auto', cmap='YlOrRd')

ax.set_xticks(range(len(DIMENSIONS)))
ax.set_xticklabels(DIMENSIONS, rotation=30, ha='right', fontsize=9)
ax.set_yticks(range(len(comp.index)))
ax.set_yticklabels(comp.index, fontsize=10)
ax.set_title('Dimension Scores by Country (sorted by Total Score)',
             fontsize=13, fontweight='bold')

vmax = comp.values.max()
for i in range(len(comp.index)):
    for j in range(len(DIMENSIONS)):
        v = comp.values[i, j]
        ax.text(j, i, f'{v:.1f}', ha='center', va='center',
                fontsize=9, color='white' if v > vmax * 0.5 else 'black')

plt.colorbar(im, ax=ax, label='Score')
plt.tight_layout()
plt.show()


## 6. Radar Profiles — All Countries Overlaid

In [ ]:
N = len(DIMENSIONS)
angles = [n / N * 2 * np.pi for n in range(N)] + [0]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

for country in df.index:
    vals = df.loc[country, DIMENSIONS].tolist()
    vals += vals[:1]
    ax.plot(angles, vals, linewidth=2, label=country, color=COLORS[country])
    ax.fill(angles, vals, alpha=0.1, color=COLORS[country])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(DIMENSIONS, fontsize=10)
ax.set_ylim(0, df[DIMENSIONS].values.max() * 1.1)
ax.set_title('Stanford AI Vibrancy — Component Radar (all 8 countries)',
             fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10)
ax.grid(True)
plt.tight_layout()
plt.show()


## 7. Radar Profiles — Per-Country Small Multiples

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9), subplot_kw=dict(projection='polar'))
axes = axes.flatten()

countries_sorted = df.sort_values('Total Score', ascending=False).index.tolist()
y_max = df[DIMENSIONS].values.max() * 1.05

for ax, country in zip(axes, countries_sorted):
    vals = df.loc[country, DIMENSIONS].tolist()
    vals += vals[:1]
    ax.plot(angles, vals, color=COLORS[country], linewidth=2)
    ax.fill(angles, vals, color=COLORS[country], alpha=0.3)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([d.replace(' ', '\n') for d in DIMENSIONS], fontsize=7)
    ax.set_ylim(0, y_max)
    total = df.loc[country, 'Total Score']
    ax.set_title(f'{country}\nTotal = {total:.1f}', fontsize=10, fontweight='bold', pad=12)

plt.suptitle('Per-Country AI Vibrancy Profile', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 8. Strengths & Weaknesses — Per-Country Bar Charts

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9), sharex=False)
axes = axes.flatten()

vmax = df[DIMENSIONS].values.max() * 1.1

for ax, country in zip(axes, countries_sorted):
    vals = df.loc[country, DIMENSIONS].sort_values()
    bars = ax.barh(vals.index, vals.values, color=COLORS[country], alpha=0.85)
    ax.set_xlim(0, vmax)
    ax.set_title(country, fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    ax.tick_params(axis='y', labelsize=8)
    for i, v in enumerate(vals.values):
        ax.text(v + 0.1, i, f'{v:.2f}', va='center', fontsize=8)

plt.suptitle('Stanford AI Vibrancy — Dimension Breakdown per Country',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 9. Each Country's Strongest and Weakest Dimensions

In [ ]:
rows = []
for country in df.index:
    vals = df.loc[country, DIMENSIONS]
    rows.append({
        'Country': country,
        'Total': df.loc[country, 'Total Score'],
        'Top Dimension': vals.idxmax(),
        'Top Score': vals.max(),
        'Bottom Dimension': vals.idxmin(),
        'Bottom Score': vals.min(),
    })

profile = pd.DataFrame(rows).sort_values('Total', ascending=False).reset_index(drop=True)
profile


## 10. Dimension Correlation Across Countries

In [ ]:
corr = df[DIMENSIONS].corr().round(2)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=30, ha='right', fontsize=9)
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index, fontsize=9)
ax.set_title('Correlation Between Vibrancy Dimensions', fontsize=12, fontweight='bold')

for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        v = corr.values[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                fontsize=8, color='white' if abs(v) > 0.6 else 'black')

plt.colorbar(im, ax=ax, label='Pearson r')
plt.tight_layout()
plt.show()


## 11. US vs China — Head-to-Head

In [ ]:
us = df.loc['United States', DIMENSIONS]
cn = df.loc['China', DIMENSIONS]

x = np.arange(len(DIMENSIONS))
width = 0.4

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width/2, us, width, label='United States', color=COLORS['United States'])
ax.bar(x + width/2, cn, width, label='China', color=COLORS['China'])
ax.set_xticks(x)
ax.set_xticklabels(DIMENSIONS, rotation=20, ha='right')
ax.set_ylabel('Score')
ax.set_title(f"US (Total = {df.loc['United States', 'Total Score']:.1f}) vs China (Total = {df.loc['China', 'Total Score']:.1f})",
             fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

for i, (u, c) in enumerate(zip(us, cn)):
    ax.text(i - width/2, u + 0.15, f'{u:.1f}', ha='center', fontsize=8)
    ax.text(i + width/2, c + 0.15, f'{c:.1f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

# Gap analysis
gap = (us - cn).sort_values(ascending=False)
print('\nUS minus China (positive = US leads, negative = China leads):')
print(gap.round(2))


## 12. Save Visualization-ready Long-format Data

In [ ]:
long_df = df[DIMENSIONS].reset_index().melt(
    id_vars='Country', var_name='Dimension', value_name='Score'
)
long_df['Total Score'] = long_df['Country'].map(df['Total Score'])

out_path = repo_root / 'data' / 'AI vibrancy tool screen shot' / 'ai_country_scores_long.csv'
long_df.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
long_df.head(10)
